# Scraping : récupération des fichiers PDF du corpus CAMille

Dans ce notebook, nous créons un robot qui ouvre la page du corpus [CAMille](https://max.de.wilde.web.ulb.be/camille/) et télécharge les 51 fichiers PDF qui y sont listés. Les fichiers sont enregistrés dans le dossier `data/camille` à la racine du dépôt.

## Imports

In [13]:
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

## Récupération de la liste des fichiers PDF

In [14]:
page_url = "https://max.de.wilde.web.ulb.be/camille/"
headers = {"User-Agent": "Mozilla/5.0 (compatible; CAMilleCorpusScraper/1.0)"}

response = requests.get(page_url, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

pdf_links = list(dict.fromkeys(
    urljoin(page_url, link["href"])
    for link in soup.find_all("a", href=True)
    if ".pdf" in link["href"].lower()
))

print(f"Nombre de fichiers PDF trouvés : {len(pdf_links)}")
pdf_links[:10]

Nombre de fichiers PDF trouvés : 51


['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB427_1920-01-10_01-00004.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB555_1836-02-08_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB638_1860-05-21_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB773_1918-11-30_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB838_1887-12-28_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB230_1903-10-16_01-0002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB427_1933-01-04_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB555_1899-01-19_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB638_1902-12-20_01-00002.pdf']

## Vérification du nombre de fichiers

In [19]:
expected_pdf_count = 51

if len(pdf_links) != expected_pdf_count:
    raise ValueError(
        f"Le TP attend {expected_pdf_count} fichiers PDF, mais {len(pdf_links)} liens ont été trouvés."
    )

print("Le nombre de liens correspond aux 51 fichiers attendus.")

Le nombre de liens correspond aux 51 fichiers attendus.


## Préparation du dossier de téléchargement

In [16]:
# Recherche de la racine du dépôt à partir du dossier courant du notebook.
current_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (current_dir, *current_dir.parents) if (path / ".git").exists()),
    current_dir,
)

output_dir = repo_root / "data" / "camille"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Les PDF seront enregistrés dans : {output_dir}")

Les PDF seront enregistrés dans : /Users/joelmufaume/tac/data/camille


## Téléchargement des fichiers PDF

In [17]:
downloaded = 0
skipped = 0

for url in pdf_links:
    filename = Path(urlparse(url).path).name
    destination = output_dir / filename

    if destination.exists() and destination.stat().st_size > 0:
        skipped += 1
        continue

    with requests.get(url, headers=headers, stream=True, timeout=60) as pdf_response:
        pdf_response.raise_for_status()
        first_chunk = next(pdf_response.iter_content(chunk_size=8192), b"")
        if not first_chunk.startswith(b"%PDF"):
            raise ValueError(f"Le contenu reçu ne semble pas être un PDF : {url}")

        with destination.open("wb") as file:
            file.write(first_chunk)
            for chunk in pdf_response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)

    downloaded += 1
    time.sleep(0.2)

print(f"Fichiers téléchargés : {downloaded}")
print(f"Fichiers déjà présents : {skipped}")

Fichiers téléchargés : 0
Fichiers déjà présents : 51


## Vérification des fichiers téléchargés

In [20]:
pdf_files = sorted(output_dir.glob("*.pdf"))
print(f"Nombre de fichiers PDF dans le dossier : {len(pdf_files)}")
print([file.name for file in pdf_files[:10]])

if len(pdf_files) != expected_pdf_count:
    raise ValueError(
        f"Le dossier devrait contenir {expected_pdf_count} PDF, mais en contient {len(pdf_files)}."
    )

Nombre de fichiers PDF dans le dossier : 51
['KB_JB230_1892-08-07_01-0003.pdf', 'KB_JB230_1903-10-16_01-0002.pdf', 'KB_JB230_1913-07-05_01-0001.pdf', 'KB_JB258_1884-09-03_01-0003.pdf', 'KB_JB258_1894-12-09_01-0003.pdf', 'KB_JB258_1906-01-09_01-0002.pdf', 'KB_JB421_1899-05-15_01-00003.pdf', 'KB_JB421_1926-10-29_01-00002.pdf', 'KB_JB421_1950-04-15_01-00004.pdf', 'KB_JB427_1920-01-10_01-00004.pdf']


## Pour en savoir plus
- Le web scraping avec Python : https://realpython.com/beautiful-soup-web-scraper-python/
- La librairie `requests` : https://requests.readthedocs.io/en/latest/
- Beautiful Soup : https://www.crummy.com/software/BeautifulSoup/bs4/doc/